<a href="https://colab.research.google.com/github/jeevan841/Innomatics-Internship/blob/main/Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-core langchain-community langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
!pip install requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.33.1
    Uninstalling requests-2.33.1:
      Successfully uninstalled requests-2.33.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [5]:
import os
import getpass
from google.colab import userdata

# Load API key
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

print("API Key Loaded:", "YES" if os.environ.get("GROQ_API_KEY") else "NO")

Enter your Groq API key: ··········
API Key Loaded: YES


In [6]:
#1. Basic LLM Call
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

response = llm.invoke("Explain LangChain in one line")
print(response.content)

LangChain is an open-source framework that enables developers to build applications powered by large language models, providing a set of tools and APIs to interact with and fine-tune these models.


In [7]:
#2. PromptTemplate usage
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple terms"
)

formatted_prompt = prompt.format(topic="LangChain")

print(formatted_prompt)
print("\n--- LLM Output ---\n")

# Using the 'llm' instance defined in previous cells
print(llm.invoke(formatted_prompt).content)

Explain LangChain in simple terms

--- LLM Output ---

LangChain is a framework that helps developers build applications using large language models (LLMs) like chatbots, virtual assistants, or other AI-powered tools. 

Imagine you have a super smart, magic notebook that can understand and respond to anything you write in it. This notebook is like a large language model. However, to make the notebook really useful, you need to teach it how to do specific tasks, like answering questions, generating text, or even controlling other devices.

LangChain is like a set of instructions that helps you teach the magic notebook (or the large language model) how to do these tasks. It provides a way to break down complex tasks into smaller, manageable pieces, and then use the language model to complete each piece.

The "chain" part of LangChain refers to the idea of linking together multiple language models, or using a single model in a series of steps, to achieve a specific goal. This allows devel

In [8]:
#3.Simple Chain

from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"topic": "AI Agents"})
print(result)

**What is an AI Agent.**
An AI agent is a computer program that can perform tasks on its own, using artificial intelligence (AI) to make decisions and take actions. Think of it like a robot that can think and act for itself.

**Key Characteristics:**

1. **Autonomy**: AI agents can work independently, without human intervention.
2. **Decision-making**: They can make decisions based on the information they have and the goals they're trying to achieve.
3. **Action**: They can take actions to achieve their goals, like moving, communicating, or solving problems.
4. **Learning**: Many AI agents can learn from their experiences and improve over time.

**Examples of AI Agents:**

1. **Virtual Assistants**: Like Siri, Alexa, or Google Assistant, which can understand voice commands and perform tasks.
2. **Self-Driving Cars**: Which use AI to navigate roads, avoid obstacles, and make decisions.
3. **Chatbots**: That can have conversations with humans, answer questions, and provide customer suppo

In [10]:
#4.Agent with tool
!pip install -q langchain langgraph langchain-groq

import os
from google.colab import userdata

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Set your Groq API key from Colab Secrets
os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY")

# Create Groq model
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Define tools

@tool
def convert_usd_to_inr(usd: float) -> str:
    """Convert USD to INR using a fixed example exchange rate of 1 USD = 83 INR."""
    inr = usd * 83
    return f"${usd:.2f} is approximately ₹{inr:.2f}"

@tool
def calculate_emi(principal: float, annual_rate: float, months: int) -> str:
    """Calculate monthly EMI for a loan using principal, annual interest rate, and loan duration in months."""
    monthly_rate = annual_rate / 12 / 100
    if monthly_rate == 0:
        emi = principal / months
    else:
        emi = principal * monthly_rate * (1 + monthly_rate) ** months / ((1 + monthly_rate) ** months - 1)
    total_payment = emi * months
    return (
        f"Monthly EMI: ₹{emi:.2f}, "
        f"Total Payment: ₹{total_payment:.2f}"
    )

tools = [convert_usd_to_inr, calculate_emi]

# Create ReAct agent
agent = create_react_agent(
    model=model,
    tools=tools
)

# Run the agent
result = agent.invoke({
    "messages": [
        (
            "human",
            "Convert 250 USD to INR and calculate the EMI for a ₹500000 loan at 10% annual interest for 24 months."
        )
    ]
})

print("\n=== Final Answer ===")
print(result["messages"][-1].content)

GROQ_API_KEY··········


/tmp/ipykernel_19913/449473640.py:45: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(



=== Final Answer ===
The conversion of 250 USD to INR is approximately ₹20750.00. The monthly EMI for a ₹500000 loan at 10% annual interest for 24 months is ₹23072.46, with a total payment of ₹553739.12.


In [11]:
#5.memory example
!pip install -qU langchain langchain-openai langchain-community
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

# 1. Define a prompt that includes a placeholder for chat history
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 2. Build the chain using LCEL (the '|' pipe operator)
chain = prompt | model | StrOutputParser()

# 3. Use a dictionary to store history (In-memory)
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 4. Wrap the chain with history management
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user_1"}}

# Interaction 1
resp1 = with_message_history.invoke({"input": "Hi, my name is jeevan."}, config=config)
print(f"Assistant: {resp1}")

# Interaction 2 (Testing memory)
resp2 = with_message_history.invoke({"input": "What is my name?"}, config=config)
print(f"Assistant: {resp2}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
Assistant: Hello Jeevan, it's nice to meet you. Is there something I can help you with or would you like to chat?
Assistant: Your name is Jeevan. You told me that when we started chatting.


In [12]:
#Vector Store Implementation (FAISS + HuggingFace)
!pip install -qU langchain-groq langchain-huggingface langchain-community docarray wikipedia sentence-transformers

# 1. Install necessary libraries for Vector Stores
!pip install -qU langchain-huggingface faiss-cpu

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Define the raw data (similar to your structure)
raw_docs = [
    Document(page_content="LangChain provides tools for building agents and managing memory.",
             metadata={"source": "intro.txt"}),
    Document(page_content="FAISS is a library for fast similarity search over dense vectors.",
             metadata={"source": "vector_info.txt"}),
    Document(page_content="The Model Context Protocol (MCP) connects AI models to local data.",
             metadata={"source": "mcp_info.txt"}),
]

# 3. Text Splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(raw_docs)
print(f"Created {len(chunks)} chunks from {len(raw_docs)} documents.")

# 4. Initialize Local Embeddings (Free, runs on Colab CPU)
# Using 'all-MiniLM-L6-v2' - lightweight and highly accurate for its size
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 5. Build the Vector Store (FAISS)
vector_store = FAISS.from_documents(chunks, embeddings)
print(" FAISS Vector store built successfully.")
print(f"Total vectors indexed: {vector_store.index.ntotal}")

# 6. Manual Similarity Search (To test the Vector Store independently)
query = "What is FAISS?"
docs = vector_store.similarity_search(query, k=1)

print("\n--- Similarity Search Result ---")
for doc in docs:
    print(f"Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content}")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.3 MB/s eta 0:00:00
Created 3 chunks from 3 documents.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 FAISS Vector store built successfully.
Total vectors indexed: 3

--- Similarity Search Result ---
Source: vector_info.txt
Content: FAISS is a library for fast similarity search over dense vectors.


In [14]:
# Updated Agent with Persistent Memory
!pip install -qU langchain-groq langgraph

import os
import getpass
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

# 1. Setup LLM
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

# 2. Define Tools
@tool
def calculator(expression: str) -> str:
    """Use this for math calculations. Input should be a mathematical expression like '25 * 8'."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

tools = [calculator]

# 3. Initialize Memory Checkpointer
memory = MemorySaver()

# 4. Create Agent with checkpointer for state persistence
agent_executor = create_react_agent(llm, tools, checkpointer=memory)

# 5. Interaction Helper
config = {"configurable": {"thread_id": "user_session_1"}}

def ask_agent(query):
    result = agent_executor.invoke({"messages": [HumanMessage(content=query)]}, config)
    return result["messages"][-1].content

# Running the specific tasks
print("Agent:", ask_agent("Hi, I am Jeevan"))
print("Agent:", ask_agent("What is 25 * 8?"))
print("Agent:", ask_agent("What is my name?"))

/tmp/ipykernel_19913/707838044.py:32: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


Agent: Hello Jeevan, it's nice to meet you. Is there something I can help you with or would you like to chat?
Agent: The answer is 200.
Agent: I don't have any information about your name. This conversation just started, and I don't have any prior knowledge about you. If you'd like to share your name, I'd be happy to chat with you.
